# Sample comparison (report-ready)

This notebook builds report-ready results across samples to support the thesis aims.

**Scope**
- Pipeline validation and rate summaries (Aim 1)
- Deletion vs insertion patterns, size distributions, and tandem duplications (Aim 2)
- Paternal age effect trends (Aim 3)

Retrotransposition-specific analyses are intentionally excluded for now.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
from Bio import SeqIO
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
    }
)

# === Configuration (edit here) ===
RESULTS_ROOT = Path("/home/peterkad/pkadmaster/indel_scanner/results")
SAMPLES = [
    "tr_plus_unmapped_diploid_v2",
    "ph_plus_unmapped_diploid_v2",
    "chk_plus_unmapped_diploid_v2",
    "da1_plus_unmapped_diploid_v2",
    "la_plus_unmapped_diploid_v2",
    "tsaed_plus_unmapped_diploid_v2",
]
# Exclude temporarily problematic samples from all analysis plots/tables.
# To include them again, remove from this set.
EXCLUDED_SAMPLES = {
    "da1_plus_unmapped_diploid_v2",
    "tsaed_plus_unmapped_diploid_v2",
}
ANALYSIS_SAMPLES = [s for s in SAMPLES if s not in EXCLUDED_SAMPLES]

SAMPLE_METADATA = pd.DataFrame(
    [
        {
            "sample": "tr_plus_unmapped_diploid_v2",
            "donor_id": "tr",
            "paternal_age": 35,
        },
        {
            "sample": "ph_plus_unmapped_diploid_v2",
            "donor_id": "ph",
            "paternal_age": 35,
        },
        {
            "sample": "chk_plus_unmapped_diploid_v2",
            "donor_id": "chk",
            "paternal_age": 25,
        },
        {
            "sample": "da1_plus_unmapped_diploid_v2",
            "donor_id": "da1",
            "paternal_age": 49,
        },
        {
            "sample": "la_plus_unmapped_diploid_v2",
            "donor_id": "la",
            "paternal_age": 25,
        },
        {
            "sample": "tsaed_plus_unmapped_diploid_v2",
            "donor_id": "tsaed",
            "paternal_age": 35,
        },
    ]
)
DONOR_MAP = SAMPLE_METADATA.set_index("sample")["donor_id"].to_dict()

DATA_DIR = Path("/home/peterkad/pkadmaster/data")
REPEAT_REGIONS_SUBDIR = "repeatregions"
FASTA_OVERRIDE: dict[str, Path] = {}


def sample_fasta_path(sample: str) -> Path:
    donor_id = DONOR_MAP.get(sample, sample)
    return DATA_DIR / donor_id / f"{donor_id}_diploid.fa"


FASTA_BY_SAMPLE = {
    sample: FASTA_OVERRIDE.get(sample, sample_fasta_path(sample))
    for sample in ANALYSIS_SAMPLES
}


def with_sample_label(df: pd.DataFrame) -> pd.DataFrame:
    if "sample" not in df.columns:
        return df
    return df.assign(sample_label=df["sample"].map(DONOR_MAP).fillna(df["sample"]))


FIGURES_DIR = Path("figures")
TABLES_DIR = Path("tables")
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)

SHORT_INS_MAX_BP = 10
SIZE_BIN_ORDER = ["1bp", "2-3bp", "4-10bp", ">10bp"]
INTERROGATED_BASES_PER_SAMPLE = 1_000_000
STR_COVERAGE_PERCENT_BY_SAMPLE = {
    "tr_plus_unmapped_diploid_v2": 1.634579,
    "ph_plus_unmapped_diploid_v2": 1.767940,
    "chk_plus_unmapped_diploid_v2": 1.715854,
    "da1_plus_unmapped_diploid_v2": 1.731574,
    "la_plus_unmapped_diploid_v2": 1.568374,
    "tsaed_plus_unmapped_diploid_v2": 1.640725,
}
STR_MOTIF_MIN_COUNT = 200

# RPTRF cache controls (for fast reruns)
RPTRF_USE_CACHE = True
RPTRF_FORCE_REBUILD = False

In [ ]:
def latest_run_dir(sample_root: Path, required_file: str | None = None) -> Path | None:
    if not sample_root.exists():
        return None
    run_dirs = [p for p in sample_root.iterdir() if p.is_dir()]
    if not run_dirs:
        return None
    if required_file:
        with_file = [p for p in run_dirs if (p / required_file).exists()]
        if with_file:
            return max(with_file, key=lambda p: p.stat().st_mtime)
    return max(run_dirs, key=lambda p: p.stat().st_mtime)


def read_tsv(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    return pd.read_csv(path, sep="\t")


def read_json(path: Path) -> dict | None:
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def save_table(df: pd.DataFrame, name: str, ext: str = "csv") -> None:
    out_path = TABLES_DIR / f"{name}.{ext}"
    if ext == "tsv":
        df.to_csv(out_path, index=False, sep="\t")
    else:
        df.to_csv(out_path, index=False)
    print(f"Wrote table: {out_path.resolve()}")


def save_figure(fig: plt.Figure, name: str) -> None:
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")


def parse_sequence_context(context: str) -> tuple[str, str, str] | None:
    match = re.match(r"^(.*)\[(.*)\](.*)$", str(context))
    if not match:
        return None
    return match.group(1), match.group(2), match.group(3)


def is_tandem_dup(prefix: str, ins: str, suffix: str) -> bool:
    if not ins:
        return False
    prefix = prefix.upper()
    suffix = suffix.upper()
    ins = ins.upper()
    return prefix.endswith(ins) or suffix.startswith(ins)


def assign_size_bin(length: int) -> str:
    if length == 1:
        return "1bp"
    if 2 <= length <= 3:
        return "2-3bp"
    if 4 <= length <= 10:
        return "4-10bp"
    return ">10bp"


def parse_rptrf_file(
    file_path: Path, include_homopolymer: bool = False
) -> list[tuple[int, int, int] | tuple[int, int, int, bool]]:
    tracts = []
    with open(file_path, "r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("*") or line.strip() == "" or line.startswith("Start"):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            try:
                start = int(parts[0])
                end = int(parts[1])
                length = int(parts[2])
            except ValueError:
                continue

            if include_homopolymer and len(parts) >= 4:
                motif_info = parts[3]
                motif_seq = None
                if "(" in motif_info and ")" in motif_info:
                    _, motif_seq = motif_info.split("(", 1)
                    motif_seq = motif_seq.rstrip(")")
                is_homopolymer = bool(motif_seq) and len(set(motif_seq)) == 1
                tracts.append((start, end, length, is_homopolymer))
            else:
                tracts.append((start, end, length))
    return tracts


def load_rptrf_tracts(repeat_regions_dir: Path) -> tuple[list, list, list[Path]]:
    str_files = sorted(repeat_regions_dir.glob("result-*.txt"))
    all_tracts: list[tuple[int, int, int]] = []
    all_tracts_with_hp: list[tuple[int, int, int, bool]] = []
    for str_file in str_files:
        all_tracts.extend(parse_rptrf_file(str_file, include_homopolymer=False))
        all_tracts_with_hp.extend(parse_rptrf_file(str_file, include_homopolymer=True))
    return all_tracts, all_tracts_with_hp, str_files


def count_assembly_bases(fasta_path: Path) -> int:
    total_bases = 0
    for record in SeqIO.parse(fasta_path, "fasta"):
        total_bases += len(record.seq)
    return total_bases


records = []
missing = []

for sample in ANALYSIS_SAMPLES:
    sample_root = RESULTS_ROOT / sample

    pipeline_run = latest_run_dir(
        sample_root, required_file="per_type_mutation_frequency.tsv"
    )
    callable_run = latest_run_dir(sample_root, required_file="callable_stats_1_10.json")

    if pipeline_run is None:
        missing.append((sample, "no_pipeline_run"))
        continue

    per_type = pipeline_run / "per_type_mutation_frequency.tsv"
    callable_bases = pipeline_run / "callable_bases.tsv"
    passed_indels = pipeline_run / "processed" / "final_passed_indels.tsv"

    callable_stats = None
    if callable_run is not None:
        callable_stats = callable_run / "callable_stats_1_10.json"

    if not per_type.exists() or not callable_bases.exists():
        missing.append((sample, str(pipeline_run)))
        continue

    records.append(
        {
            "sample": sample,
            "pipeline_run": pipeline_run,
            "callable_run": callable_run,
            "per_type": per_type,
            "callable_bases": callable_bases,
            "passed_indels": passed_indels if passed_indels.exists() else None,
            "callable_stats": callable_stats if callable_stats and callable_stats.exists() else None,
        }
    )

path_debug = pd.DataFrame(
    [
        {
            "sample": r["sample"],
            "pipeline_run": str(r["pipeline_run"]),
            "callable_run": str(r["callable_run"]) if r["callable_run"] else "missing",
            "callable_stats": str(r["callable_stats"]) if r["callable_stats"] else "missing",
        }
        for r in records
    ]
)
save_table(path_debug, "callable_stats_paths", ext="tsv")

pd.DataFrame(records), pd.DataFrame(missing, columns=["sample", "issue"])

(Empty DataFrame
 Columns: []
 Index: [],
                            sample       issue
 0     tr_plus_unmapped_diploid_v2  no_run_dir
 1     ph_plus_unmapped_diploid_v2  no_run_dir
 2    chk_plus_unmapped_diploid_v2  no_run_dir
 3    da1_plus_unmapped_diploid_v2  no_run_dir
 4     la_plus_unmapped_diploid_v2  no_run_dir
 5  tsaed_plus_unmapped_diploid_v2  no_run_dir)

In [ ]:
def load_per_type(path: Path, sample: str) -> pd.DataFrame:
    df = read_tsv(path)
    if df is None:
        return pd.DataFrame()
    df["sample"] = sample
    return df


def load_callable(path: Path, sample: str) -> pd.DataFrame:
    df = read_tsv(path)
    if df is None:
        return pd.DataFrame()
    df["sample"] = sample
    return df


def load_passed_indels(path: Path | None, sample: str) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    df = read_tsv(path)
    if df is None:
        return pd.DataFrame()
    df["sample"] = sample
    return df


per_type_dfs = []
callable_dfs = []
passed_dfs = []

for rec in records:
    sample = rec["sample"]
    per_type_dfs.append(load_per_type(rec["per_type"], sample))
    callable_dfs.append(load_callable(rec["callable_bases"], sample))
    passed_dfs.append(load_passed_indels(rec["passed_indels"], sample))

per_type_all = pd.concat(per_type_dfs, ignore_index=True) if per_type_dfs else pd.DataFrame()
callable_all = pd.concat(callable_dfs, ignore_index=True) if callable_dfs else pd.DataFrame()
passed_all = pd.concat(passed_dfs, ignore_index=True) if passed_dfs else pd.DataFrame()

per_type_all.head()

## RPTRF repeat-region summary

This section summarizes tandem-repeat coverage and homopolymer content across samples using RPTRF repeat-region outputs.

In [ ]:
if repeat_summary.empty:
    print("No RPTRF repeat-region summaries available for averages.")
else:
    numeric_cols = repeat_summary.select_dtypes(include=["number"]).columns
    repeat_averages = repeat_summary[numeric_cols].mean(numeric_only=True).to_frame().T
    repeat_averages.insert(0, "stat", "mean")
    save_table(repeat_averages, "rptrf_repeat_summary_averages", ext="tsv")
    repeat_averages

In [ ]:
rptrf_cache_path = TABLES_DIR / "rptrf_repeat_summary.tsv"
rptrf_missing_cache_path = TABLES_DIR / "rptrf_repeat_summary_missing.tsv"

repeat_summary = pd.DataFrame()
repeat_missing_df = pd.DataFrame()

use_cached_rptrf = (
    RPTRF_USE_CACHE
    and not RPTRF_FORCE_REBUILD
    and rptrf_cache_path.exists()
)

if use_cached_rptrf:
    repeat_summary = read_tsv(rptrf_cache_path)
    if repeat_summary is None:
        repeat_summary = pd.DataFrame()
    if rptrf_missing_cache_path.exists():
        repeat_missing_df = read_tsv(rptrf_missing_cache_path)
        if repeat_missing_df is None:
            repeat_missing_df = pd.DataFrame()
    else:
        repeat_missing_df = pd.DataFrame()

    if not repeat_summary.empty:
        repeat_summary = with_sample_label(repeat_summary)
        sample_order = [DONOR_MAP.get(s, s) for s in ANALYSIS_SAMPLES]
        repeat_summary["sample_label"] = pd.Categorical(
            repeat_summary["sample_label"], categories=sample_order, ordered=True
        )
        repeat_summary = repeat_summary.sort_values("sample_label")
else:
    repeat_stats_rows = []
    repeat_missing = []

    for sample in ANALYSIS_SAMPLES:
        donor_id = DONOR_MAP.get(sample, sample)
        repeat_dir = DATA_DIR / donor_id / REPEAT_REGIONS_SUBDIR
        fasta_path = FASTA_BY_SAMPLE.get(sample)

        issues = []
        if not repeat_dir.exists():
            issues.append("missing_repeatregions")
        if fasta_path is None or not fasta_path.exists():
            issues.append("missing_fasta")
        if issues:
            repeat_missing.append({"sample": sample, "issue": ";".join(issues)})
            continue

        all_tracts, all_tracts_with_hp, str_files = load_rptrf_tracts(repeat_dir)
        if not all_tracts:
            repeat_missing.append({"sample": sample, "issue": "no_repeat_tracts"})
            continue

        total_str_bases = sum(tract[2] for tract in all_tracts)
        total_assembly_bases = count_assembly_bases(fasta_path)
        if total_assembly_bases == 0:
            repeat_missing.append({"sample": sample, "issue": "zero_assembly_bases"})
            continue

        bp_per_repeat_region = total_assembly_bases / len(all_tracts)
        repeat_regions_per_mbp = len(all_tracts) / (total_assembly_bases / 1_000_000)
        repeat_coverage_percent = (total_str_bases / total_assembly_bases) * 100

        hp_bases = sum(tract[2] for tract in all_tracts_with_hp if tract[3])
        hp_region_count = sum(1 for tract in all_tracts_with_hp if tract[3])
        hp_pct_of_repeat_bases = (
            (hp_bases / total_str_bases) * 100 if total_str_bases else 0.0
        )
        hp_pct_of_regions = (
            (hp_region_count / len(all_tracts)) * 100 if len(all_tracts) else 0.0
        )

        non_hp_bases = total_str_bases - hp_bases
        non_hp_region_count = len(all_tracts) - hp_region_count

        repeat_stats_rows.append(
            {
                "sample": sample,
                "donor_id": donor_id,
                "repeat_regions_dir": str(repeat_dir),
                "fasta_path": str(fasta_path),
                "repeat_region_count": len(all_tracts),
                "repeat_bases": total_str_bases,
                "repeat_coverage_percent": repeat_coverage_percent,
                "bp_per_repeat_region": bp_per_repeat_region,
                "repeat_regions_per_mbp": repeat_regions_per_mbp,
                "homopolymer_region_count": hp_region_count,
                "homopolymer_bases": hp_bases,
                "homopolymer_pct_of_repeat_bases": hp_pct_of_repeat_bases,
                "homopolymer_pct_of_regions": hp_pct_of_regions,
                "non_homopolymer_region_count": non_hp_region_count,
                "non_homopolymer_bases": non_hp_bases,
                "repeat_files": len(str_files),
            }
        )

    repeat_summary = pd.DataFrame(repeat_stats_rows)
    if not repeat_summary.empty:
        repeat_summary = with_sample_label(repeat_summary)
        sample_order = [DONOR_MAP.get(s, s) for s in ANALYSIS_SAMPLES]
        repeat_summary["sample_label"] = pd.Categorical(
            repeat_summary["sample_label"], categories=sample_order, ordered=True
        )
        repeat_summary = repeat_summary.sort_values("sample_label")
        save_table(repeat_summary, "rptrf_repeat_summary", ext="tsv")

    repeat_missing_df = pd.DataFrame(repeat_missing)
    save_table(repeat_missing_df, "rptrf_repeat_summary_missing", ext="tsv")

repeat_summary, repeat_missing_df

In [ ]:
if repeat_summary.empty:
    print("No RPTRF repeat-region summaries available.")
else:
    # Keep RPTRF as compact tables only (no plots).
    rptrf_compact = repeat_summary[
        [
            "sample",
            "sample_label",
            "repeat_region_count",
            "repeat_bases",
            "repeat_coverage_percent",
            "repeat_regions_per_mbp",
            "homopolymer_region_count",
            "homopolymer_bases",
        ]
    ].copy()
    save_table(rptrf_compact, "rptrf_repeat_summary_compact", ext="tsv")
    display(rptrf_compact)


In [ ]:
## Aim 1 — Pipeline validation and basic rate summaries

Goal: confirm the pipeline outputs are consistent across samples and produce stable callable bases and rate estimates.

In [ ]:
if per_type_all.empty:
    print("No per-type data found. Check RESULTS_ROOT/ANALYSIS_SAMPLES.")
else:
    # Keep Aim 1 summary compact: table only.
    indel_summary = per_type_all.groupby("sample", as_index=False).agg(
        total_indels=("count", "sum")
    )
    if "unique_sites" in per_type_all.columns:
        unique_summary = per_type_all.groupby("sample", as_index=False)["unique_sites"].sum()
        indel_summary = indel_summary.merge(unique_summary, on="sample", how="left")
        indel_summary = indel_summary.rename(columns={"unique_sites": "total_unique_sites"})

    indel_summary = with_sample_label(indel_summary)
    save_table(indel_summary, "aim1_indel_summary", ext="tsv")
    display(indel_summary)


In [ ]:
# Non-STR 1–10bp per-class mutation rates (heatmap)
non_str_rates = per_type_all.copy()
non_str_rates = non_str_rates[non_str_rates["str_class"] == "non_STR"].copy()
non_str_rates["size_bp"] = non_str_rates["mutation_type"].str.extract(r"len_(\d+)bp")[0]
non_str_rates = non_str_rates.dropna(subset=["size_bp"]).copy()
non_str_rates["size_bp"] = non_str_rates["size_bp"].astype(int)
non_str_rates = non_str_rates[non_str_rates["size_bp"] <= 10].copy()
non_str_rates["indel_type"] = non_str_rates["mutation_type"].str.extract(r"^(ins|del)")

non_str_rates = with_sample_label(non_str_rates)

rate_ins = non_str_rates[non_str_rates["indel_type"] == "ins"].copy()
rate_del = non_str_rates[non_str_rates["indel_type"] == "del"].copy()

rate_ins = rate_ins.sort_values(["size_bp"], ascending=False)
rate_del = rate_del.sort_values(["size_bp"], ascending=False)

heatmap_ins = rate_ins.pivot(
    index="size_bp",
    columns="sample_label",
    values="frequency",
)
heatmap_del = rate_del.pivot(
    index="size_bp",
    columns="sample_label",
    values="frequency",
)

heatmap_ins = heatmap_ins.where(heatmap_ins > 0)
heatmap_del = heatmap_del.where(heatmap_del > 0)

combined_min = np.nanmin(
    pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
)
combined_max = np.nanmax(
    pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
)

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(6.8, 6.6),
    sharex=True,
    gridspec_kw={"hspace": 0.06},
)
fig.subplots_adjust(right=0.86)
cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

sns.heatmap(
    heatmap_del,
    cmap="viridis",
    linewidths=0.2,
    linecolor="white",
    norm=LogNorm(vmin=combined_min, vmax=combined_max),
    annot=True,
    fmt=".1e",
    annot_kws={"fontsize": 7},
    cbar=False,
    ax=axes[0],
)
axes[0].set_title("")
axes[0].set_ylabel("Deletions (bp)")
axes[0].set_xlabel("")

sns.heatmap(
    heatmap_ins,
    cmap="viridis",
    linewidths=0.2,
    linecolor="white",
    norm=LogNorm(vmin=combined_min, vmax=combined_max),
    annot=True,
    fmt=".1e",
    annot_kws={"fontsize": 7},
    cbar=True,
    cbar_ax=cbar_ax,
    cbar_kws={"label": "Mutation rate (per base, log scale)"},
    ax=axes[1],
)
axes[1].set_title("")
axes[1].set_ylabel("Insertions (bp)")
axes[1].set_xlabel("Sample")

fig.suptitle("Non-STR 1–10bp per-class mutation rates")
fig.tight_layout(rect=[0, 0, 0.86, 0.96])
save_figure(fig, "aim1_non_str_per_class_rates_heatmap")
plt.close(fig)


In [ ]:
# Supplementary panel: non-STR insertion/deletion rates per sample
if per_type_all.empty:
    print("No per-type data available for supplementary non-STR rate panel.")
else:
    fig2a_non_str = per_type_all.copy()
    fig2a_non_str["indel_type"] = fig2a_non_str["mutation_type"].str.extract(r"^(ins|del)")
    fig2a_non_str["size_bp"] = pd.to_numeric(
        fig2a_non_str["mutation_type"].str.extract(r"len_(\d+)bp")[0],
        errors="coerce",
    )
    fig2a_non_str = fig2a_non_str[
        (fig2a_non_str["str_class"] == "non_STR")
        & (fig2a_non_str["size_bp"].between(1, 10))
        & (fig2a_non_str["indel_type"].isin(["ins", "del"]))
    ].copy()

    if fig2a_non_str.empty:
        print("No non-STR 1-10bp rows available for supplementary panel.")
    else:
        fig2a_non_str = (
            fig2a_non_str.groupby(["sample", "indel_type"], as_index=False)[["count", "callable_bases"]]
            .sum()
        )
        fig2a_non_str["rate"] = fig2a_non_str["count"] / fig2a_non_str["callable_bases"]
        fig2a_non_str = with_sample_label(fig2a_non_str)

        sample_order = [DONOR_MAP.get(s, s) for s in ANALYSIS_SAMPLES]
        fig2a_non_str["sample_label"] = pd.Categorical(
            fig2a_non_str["sample_label"], categories=sample_order, ordered=True
        )
        fig2a_non_str = fig2a_non_str.sort_values(["sample_label", "indel_type"])

        save_table(
            fig2a_non_str[["sample", "sample_label", "indel_type", "count", "callable_bases", "rate"]],
            "aim1_suppl_non_str_ins_del_rates",
            ext="tsv",
        )

        fig, ax = plt.subplots(figsize=(6.4, 3.6))
        sns.barplot(
            data=fig2a_non_str,
            x="sample_label",
            y="rate",
            hue="indel_type",
            hue_order=["del", "ins"],
            palette=sns.color_palette("viridis", 2),
            ax=ax,
        )

        min_rate = fig2a_non_str["rate"].min()
        max_rate = fig2a_non_str["rate"].max()
        ax.set_yscale("log")
        ax.set_ylim(min_rate / 1.8, max_rate * 1.8)

        for p in ax.patches:
            h = p.get_height()
            if np.isfinite(h) and h > 0:
                ax.annotate(
                    f"{h:.1e}",
                    (p.get_x() + p.get_width() / 2, h),
                    ha="center",
                    va="bottom",
                    fontsize=7,
                    xytext=(0, 2),
                    textcoords="offset points",
                    rotation=90,
                )

        ax.set_xlabel("Sample")
        ax.set_ylabel("Rate (per base, log10 scale)")
        ax.legend(title="Type", frameon=False)
        fig.suptitle("Supplementary: Non-STR 1-10 bp insertion/deletion rates")
        fig.tight_layout()
        save_figure(fig, "aim1_suppl_non_str_ins_del_rates")
        plt.close(fig)

In [ ]:
# Aim 1 summary panel: pooled non-STR 1-10bp DNM rates by class
if per_type_all.empty:
    print("No per-type data available for class-summary panel.")
else:
    class_summary = per_type_all.copy()
    class_summary["indel_type"] = class_summary["mutation_type"].str.extract(r"^(ins|del)")
    class_summary["size_bp"] = pd.to_numeric(
        class_summary["mutation_type"].str.extract(r"len_(\d+)bp")[0],
        errors="coerce",
    )
    class_summary = class_summary[
        (class_summary["str_class"] == "non_STR")
        & (class_summary["size_bp"].between(1, 10))
        & (class_summary["indel_type"].isin(["ins", "del"]))
    ].copy()

    if class_summary.empty:
        print("No non-STR 1-10bp rows available for class-summary panel.")
    else:
        pooled = (
            class_summary.groupby(["mutation_type", "indel_type", "size_bp"], as_index=False)[["count", "callable_bases"]]
            .sum()
        )
        pooled["rate"] = pooled["count"] / pooled["callable_bases"]
        pooled = pooled.sort_values(["indel_type", "size_bp"], ascending=[True, False])

        save_table(pooled, "aim1_non_str_per_class_rates_pooled", ext="tsv")

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(6.8, 6.0),
            sharex=True,
            gridspec_kw={"hspace": 0.08},
        )
        for ax, indel, title in zip(axes, ["del", "ins"], ["Deletions", "Insertions"]):
            p = pooled[pooled["indel_type"] == indel].copy()
            p = p.sort_values("size_bp", ascending=False)
            labels = p["size_bp"].astype(int).astype(str)
            ax.barh(labels, p["rate"], color=sns.color_palette("viridis", 6)[3])
            ax.invert_yaxis()
            ax.set_xscale("log")
            ax.set_ylabel(title)
            for yy, vv in zip(labels, p["rate"]):
                ax.text(vv * 1.08, yy, f"{vv:.1e}", va="center", fontsize=7)

        axes[1].set_xlabel("Rate (per base, log10 scale)")
        fig.suptitle("Pooled non-STR 1-10 bp DNM rates by class")
        fig.tight_layout()
        save_figure(fig, "aim1_non_str_per_class_rates_summary")
        plt.close(fig)

**Primary Figure caption (Aim 1, class summary panel):** Pooled non-STR 1-10 bp DNM rates by class. For each class, counts and callable bases are summed across available samples, and class rate is computed as pooled count divided by pooled callable bases; deletions and insertions are shown in separate panels on a log10 x-axis.

**Interpretation note:** This provides a compact class-level summary of DNM rates without sample-by-sample clutter. Sample-level heterogeneity remains available in the annotated per-class heatmap.

**Figure caption (Aim 1, adapted from Fig. 2a):** Non-STR 1-10 bp insertion and deletion mutation rates across samples. Bars report per-sample rates computed as class-specific indel counts divided by class-specific callable bases, aggregated over non-STR 1-10 bp classes.

**Comparability note:** Comparisons are valid across samples within indel type because each rate uses the same class scope and class-specific callable denominator. Insertion-versus-deletion totals should not be treated as sharing one additive callable space across classes.

In [ ]:
display(
    Markdown(
        """
**Interpretation (Aim 1):**
The primary class-summary panel reports pooled non-STR 1-10 bp DNM rates by class, while the annotated heatmap provides sample-level per-class rates.

**Comparability notes:**
- Per-class rates in `per_type_mutation_frequency.tsv` are normalized by class-specific callable bases.
- Heatmap comparisons are valid across samples within the same class.
- Class totals should not be treated as sharing one additive callable denominator.
"""
    )
)


In [ ]:
## Aim 2 — Indel mechanisms (no retrotransposition)

Goal: compare insertion vs deletion rates, size distributions, and tandem duplication signature in short insertions.

In [ ]:
if per_type_all.empty:
    print("No per-type data available for Aim 2.")
else:
    per_type_all = per_type_all.copy()
    per_type_all["indel_type"] = per_type_all["mutation_type"].str.extract(r"^(ins|del)")

    non_str_rates = per_type_all[per_type_all["str_class"] == "non_STR"].copy()
    non_str_rates["size_bp"] = non_str_rates["mutation_type"].str.extract(
        r"len_(\d+)bp"
    )[0]
    non_str_rates = non_str_rates.dropna(subset=["size_bp"]).copy()
    non_str_rates["size_bp"] = non_str_rates["size_bp"].astype(int)
    non_str_rates = non_str_rates[non_str_rates["size_bp"] <= 10].copy()
    non_str_rates["indel_type"] = non_str_rates["mutation_type"].str.extract(
        r"^(ins|del)"
    )
    non_str_rates = with_sample_label(non_str_rates)

    non_str_summary = (
        non_str_rates.groupby(["sample", "indel_type"], as_index=False)
        .agg(
            count=("count", "sum"),
            callable_bases=("callable_bases", "sum"),
        )
        .assign(rate=lambda df: df["count"] / df["callable_bases"])
    )
    non_str_summary = with_sample_label(non_str_summary)

    non_str_summary_wide = non_str_summary.pivot(
        index=["sample", "sample_label"],
        columns="indel_type",
        values=["count", "callable_bases", "rate"],
    )
    non_str_summary_wide.columns = [
        f"{indel}_{metric}" for metric, indel in non_str_summary_wide.columns
    ]
    non_str_summary_wide = non_str_summary_wide.reset_index()

    save_table(
        non_str_summary_wide,
        "aim2_non_str_ins_del_summary",
        ext="tsv",
    )

    non_str_summary = (
        non_str_rates.groupby(["sample", "indel_type"], as_index=False)
        .agg(
            count=("count", "sum"),
            callable_bases=("callable_bases", "sum"),
        )
        .assign(rate=lambda df: df["count"] / df["callable_bases"])
    )
    non_str_summary = with_sample_label(non_str_summary)

    non_str_summary_wide = non_str_summary.pivot(
        index=["sample", "sample_label"],
        columns="indel_type",
        values=["count", "callable_bases", "rate"],
    )
    non_str_summary_wide.columns = [
        f"{indel}_{metric}" for metric, indel in non_str_summary_wide.columns
    ]
    non_str_summary_wide = non_str_summary_wide.reset_index()

    save_table(
        non_str_summary_wide,
        "aim2_non_str_ins_del_summary",
        ext="tsv",
    )

    rate_stats = (
        non_str_rates.groupby(["sample", "sample_label", "indel_type"], as_index=False)
        .agg(
            rate_mean=("frequency", "mean"),
            rate_median=("frequency", "median"),
            rate_min=("frequency", "min"),
            rate_max=("frequency", "max"),
        )
    )
    rate_stats_wide = rate_stats.pivot(
        index=["sample", "sample_label"],
        columns="indel_type",
        values=["rate_mean", "rate_median", "rate_min", "rate_max"],
    )
    rate_stats_wide.columns = [
        f"{indel}_{metric}" for metric, indel in rate_stats_wide.columns
    ]
    rate_stats_wide = rate_stats_wide.reset_index()
    save_table(
        rate_stats_wide,
        "aim2_non_str_rate_summary_stats",
        ext="tsv",
    )
    display(rate_stats_wide)

    class_stats = (
        non_str_rates.groupby(["mutation_type", "indel_type"], as_index=False)
        .agg(
            rate_min=("frequency", "min"),
            rate_median=("frequency", "median"),
            rate_mean=("frequency", "mean"),
            rate_max=("frequency", "max"),
        )
    )
    class_stats["rate_span_fold"] = class_stats["rate_max"] / class_stats["rate_min"]
    save_table(
        class_stats,
        "aim2_non_str_rate_class_span",
        ext="tsv",
    )
    display(class_stats)

    sample_stats = (
        non_str_rates.groupby(["sample", "sample_label"], as_index=False)
        .agg(
            rate_min=("frequency", "min"),
            rate_median=("frequency", "median"),
            rate_mean=("frequency", "mean"),
            rate_max=("frequency", "max"),
        )
    )
    sample_stats["rate_span_fold"] = sample_stats["rate_max"] / sample_stats["rate_min"]
    save_table(
        sample_stats,
        "aim2_non_str_rate_sample_span",
        ext="tsv",
    )
    display(sample_stats)

    rate_ins = non_str_rates[non_str_rates["indel_type"] == "ins"].copy()
    rate_del = non_str_rates[non_str_rates["indel_type"] == "del"].copy()
    rate_ins = rate_ins.sort_values(["size_bp"], ascending=False)
    rate_del = rate_del.sort_values(["size_bp"], ascending=False)

    heatmap_ins = rate_ins.pivot(
        index="size_bp",
        columns="sample_label",
        values="frequency",
    )
    heatmap_del = rate_del.pivot(
        index="size_bp",
        columns="sample_label",
        values="frequency",
    )

    heatmap_ins = heatmap_ins.where(heatmap_ins > 0)
    heatmap_del = heatmap_del.where(heatmap_del > 0)

    combined_min = np.nanmin(
        pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
    )
    combined_max = np.nanmax(
        pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
    )

    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(6.8, 6.6),
        sharex=True,
        gridspec_kw={"hspace": 0.06},
    )
    fig.subplots_adjust(right=0.86)
    cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

    sns.heatmap(
        heatmap_del,
        cmap="viridis",
        linewidths=0.2,
        linecolor="white",
        norm=LogNorm(vmin=combined_min, vmax=combined_max),
        cbar=False,
        ax=axes[0],
    )
    axes[0].set_title("")
    axes[0].set_ylabel("Deletions (bp)")
    axes[0].set_xlabel("")

    sns.heatmap(
        heatmap_ins,
        cmap="viridis",
        linewidths=0.2,
        linecolor="white",
        norm=LogNorm(vmin=combined_min, vmax=combined_max),
        cbar=True,
        cbar_ax=cbar_ax,
        cbar_kws={"label": "Mutation rate (per base, log scale)"},
        ax=axes[1],
    )
    axes[1].set_title("")
    axes[1].set_ylabel("Insertions (bp)")
    axes[1].set_xlabel("Sample")

    fig.suptitle("Non-STR 1–10bp mutation rates (per class)")
    fig.tight_layout(rect=[0, 0, 0.86, 0.96])
    save_figure(fig, "aim2_non_str_rate_heatmap_1_10")
    plt.close(fig)

    if not passed_all.empty and "length" in passed_all.columns and "type" in passed_all.columns:
        passed_all = passed_all.copy()
        passed_all = passed_all[passed_all["type"].isin(["ins", "del"])].copy()
        passed_all["length"] = pd.to_numeric(passed_all["length"], errors="coerce")
        passed_all = passed_all.dropna(subset=["length"]) 
        passed_all["length"] = passed_all["length"].astype(int)
        passed_all["size_bin"] = passed_all["length"].apply(assign_size_bin)

        if {"in_STR_region", "in_STR"}.issubset(passed_all.columns):
            non_str_mask = ~(passed_all["in_STR_region"] & passed_all["in_STR"])
        elif "in_STR" in passed_all.columns:
            non_str_mask = ~passed_all["in_STR"]
        else:
            non_str_mask = pd.Series([True] * len(passed_all), index=passed_all.index)

        passed_non_str = passed_all[non_str_mask].copy()

        size_counts = (
            passed_non_str.groupby(["sample", "type", "size_bin"], as_index=False)
            .size()
            .rename(columns={"size": "count"})
        )
        size_counts["fraction"] = size_counts["count"] / size_counts.groupby(
            ["sample", "type"]
        )["count"].transform("sum")
        size_counts["size_bin"] = pd.Categorical(
            size_counts["size_bin"], categories=SIZE_BIN_ORDER, ordered=True
        )
        size_counts = size_counts.sort_values(["type", "size_bin"])
        save_table(size_counts, "aim2_size_distribution_non_str")

        size_counts = with_sample_label(size_counts)
        ins_size = size_counts[size_counts["type"] == "ins"].copy()
        del_size = size_counts[size_counts["type"] == "del"].copy()

        heatmap_ins = ins_size.pivot(
            index="size_bin",
            columns="sample_label",
            values="fraction",
        )
        heatmap_del = del_size.pivot(
            index="size_bin",
            columns="sample_label",
            values="fraction",
        )

        combined_min = np.nanmin(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )
        combined_max = np.nanmax(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(6.8, 6.6),
            sharex=True,
            gridspec_kw={"hspace": 0.06},
        )
        fig.subplots_adjust(right=0.86)
        cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

        sns.heatmap(
            heatmap_del,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=False,
            ax=axes[0],
        )
        axes[0].set_title("")
        axes[0].set_ylabel("Deletions")
        axes[0].set_xlabel("")

        sns.heatmap(
            heatmap_ins,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=True,
            cbar_ax=cbar_ax,
            cbar_kws={"label": "Fraction within type"},
            ax=axes[1],
        )
        axes[1].set_title("")
        axes[1].set_ylabel("Insertions")
        axes[1].set_xlabel("Sample")

        fig.suptitle("Non-STR size distribution (fraction within type)")
        fig.tight_layout(rect=[0, 0, 0.86, 0.96])
        save_figure(fig, "aim2_size_distribution_non_str")
        plt.close(fig)

        context_col = "[sequence]_context"
        if context_col in passed_non_str.columns:
            ins_df = passed_non_str[
                (passed_non_str["type"] == "ins")
                & (passed_non_str["length"] <= SHORT_INS_MAX_BP)
            ].copy()
            if not ins_df.empty:
                parsed = ins_df[context_col].apply(parse_sequence_context)
                ins_df["prefix"] = parsed.apply(lambda x: x[0] if x else "")
                ins_df["ins_seq"] = parsed.apply(lambda x: x[1] if x else "")
                ins_df["suffix"] = parsed.apply(lambda x: x[2] if x else "")
                ins_df["is_tandem_dup"] = ins_df.apply(
                    lambda r: is_tandem_dup(r["prefix"], r["ins_seq"], r["suffix"]),
                    axis=1,
                )

                tandem_summary = (
                    ins_df.groupby("sample", as_index=False)
                    .agg(
                        short_insertions=("is_tandem_dup", "size"),
                        tandem_dups=("is_tandem_dup", "sum"),
                    )
                )
                tandem_summary["fraction_tandem"] = (
                    tandem_summary["tandem_dups"] / tandem_summary["short_insertions"]
                )
                save_table(tandem_summary, "aim2_tandem_duplications")

                tandem_summary = with_sample_label(tandem_summary)

                fig, ax = plt.subplots(figsize=(6.2, 3.4))
                sns.barplot(
                    data=tandem_summary,
                    x="sample_label",
                    y="fraction_tandem",
                    ax=ax,
                )
                ax.set_ylabel("Fraction tandem duplication")
                ax.set_xlabel("Sample")
                fig.suptitle("Short insertions consistent with tandem duplication")
                fig.tight_layout()
                save_figure(fig, "aim2_tandem_dup_fraction")
                plt.close(fig)

        non_str_rates.head()

**Figure caption (Aim 2, adapted from Fig. 3a):** STR motif-length mutation-rate profiles by sample. Points and lines show mean per-class STR mutation rates from `per_type_mutation_frequency.tsv` as a function of motif length, plotted separately for deletions and insertions on a log-scaled y-axis.

**Comparability note:** Within each motif length and indel type, cross-sample comparisons are valid because rates are normalized by class-specific callable bases. Across motif lengths, this figure is interpreted as profile shape and relative scaling, not additive callable space.

In [ ]:
# Fig. 3-adapted: STR mutation rate vs motif length by sample
if per_type_all.empty:
    print("No per-type data available for STR motif-length rate panel.")
else:
    tr_rate = per_type_all.copy()
    tr_rate = tr_rate[tr_rate["str_class"] == "STR_motif"].copy()
    tr_rate["motif_len"] = pd.to_numeric(
        tr_rate["mutation_type"].str.extract(r"motif_(\d+)bp")[0], errors="coerce"
    )
    tr_rate["indel_type"] = tr_rate["mutation_type"].str.extract(r"^(ins|del)")
    tr_rate = tr_rate.dropna(subset=["motif_len", "frequency", "indel_type"]).copy()
    tr_rate["motif_len"] = tr_rate["motif_len"].astype(int)
    tr_rate = tr_rate[tr_rate["frequency"] > 0].copy()
    tr_rate = with_sample_label(tr_rate)

    if tr_rate.empty:
        print("No positive STR motif-class rates available for Fig. 3-adapted panel.")
    else:
        tr_rate_summary = (
            tr_rate.groupby(["sample", "sample_label", "indel_type", "motif_len"], as_index=False)["frequency"]
            .mean()
            .rename(columns={"frequency": "rate"})
        )
        save_table(tr_rate_summary, "aim2_fig3_tr_rate_by_motif_length", ext="tsv")

        sample_order = [DONOR_MAP.get(s, s) for s in ANALYSIS_SAMPLES]
        palette = dict(zip(sample_order, sns.color_palette("viridis", len(sample_order))))

        fig, axes = plt.subplots(
            nrows=1,
            ncols=2,
            figsize=(8.4, 3.8),
            sharey=True,
            gridspec_kw={"wspace": 0.18},
        )

        for ax, indel_type, panel_title in zip(axes, ["del", "ins"], ["Deletions", "Insertions"]):
            panel = tr_rate_summary[tr_rate_summary["indel_type"] == indel_type].copy()
            for sample_label in sample_order:
                s_df = panel[panel["sample_label"] == sample_label].sort_values("motif_len")
                if s_df.empty:
                    continue
                ax.plot(
                    s_df["motif_len"],
                    s_df["rate"],
                    marker="o",
                    markersize=3,
                    linewidth=1.0,
                    color=palette[sample_label],
                    label=sample_label,
                    alpha=0.95,
                )
            ax.set_yscale("log")
            ax.set_xlabel("Motif length (bp)")
            ax.set_title(panel_title)
            ax.grid(True, axis="y", linestyle="--", linewidth=0.4, alpha=0.45)

        axes[0].set_ylabel("Rate")
        handles, labels = axes[1].get_legend_handles_labels()
        if handles:
            uniq = dict(zip(labels, handles))
            fig.legend(
                uniq.values(),
                uniq.keys(),
                title="Sample",
                loc="center left",
                bbox_to_anchor=(1.01, 0.5),
                frameon=False,
            )

        fig.suptitle("STR mutation rates by motif length and sample")
        fig.tight_layout(rect=[0, 0, 0.86, 0.94])
        save_figure(fig, "aim2_fig3_tr_rate_by_motif_length")
        plt.close(fig)

In [ ]:
# STR motif-class reporting (adaptive bins)
if per_type_all.empty or callable_all.empty:
    print("No per-type or callable data available for STR reporting.")
else:
    def plot_stacked_heatmap(
        df: pd.DataFrame,
        value_col: str,
        title: str,
        filename: str,
        cbar_label: str,
        index_col: str,
        index_order: list[str] | None = None,
        log_scale: bool = False,
    ) -> None:
        rate_ins = df[df["indel_type"] == "ins"].copy()
        rate_del = df[df["indel_type"] == "del"].copy()

        heatmap_ins = rate_ins.pivot(
            index=index_col,
            columns="sample_label",
            values=value_col,
        )
        heatmap_del = rate_del.pivot(
            index=index_col,
            columns="sample_label",
            values=value_col,
        )
        if index_order:
            heatmap_ins = heatmap_ins.reindex(index_order)
            heatmap_del = heatmap_del.reindex(index_order)

        if log_scale:
            heatmap_ins = heatmap_ins.where(heatmap_ins > 0)
            heatmap_del = heatmap_del.where(heatmap_del > 0)

        combined = pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        if combined.empty:
            return
        combined_min = np.nanmin(combined)
        combined_max = np.nanmax(combined)

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(6.8, 6.6),
            sharex=True,
            gridspec_kw={"hspace": 0.06},
        )
        fig.subplots_adjust(right=0.86)
        cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

        heatmap_kwargs = {
            "cmap": "viridis",
            "linewidths": 0.2,
            "linecolor": "white",
            "cbar": False,
            "ax": axes[0],
        }
        if log_scale:
            heatmap_kwargs["norm"] = LogNorm(vmin=combined_min, vmax=combined_max)
        else:
            heatmap_kwargs["vmin"] = combined_min
            heatmap_kwargs["vmax"] = combined_max

        sns.heatmap(heatmap_del, **heatmap_kwargs)
        axes[0].set_title("")
        axes[0].set_ylabel("Deletions (motif bin)")
        axes[0].set_xlabel("")

        heatmap_kwargs.update(
            {
                "cbar": True,
                "cbar_ax": cbar_ax,
                "cbar_kws": {"label": cbar_label},
                "ax": axes[1],
            }
        )
        sns.heatmap(heatmap_ins, **heatmap_kwargs)
        axes[1].set_title("")
        axes[1].set_ylabel("Insertions (motif bin)")
        axes[1].set_xlabel("Sample")

        fig.suptitle(title)
        fig.tight_layout(rect=[0, 0, 0.86, 0.96])
        save_figure(fig, filename)
        plt.close(fig)

    def make_motif_bins(
        counts: pd.Series, min_count: int
    ) -> tuple[dict[int, str], list[str]]:
        lengths = counts.sort_index()
        bins: list[list[int]] = []
        current: list[int] = []
        running = 0
        for motif_len, count in lengths.items():
            current.append(int(motif_len))
            running += int(count)
            if running >= min_count:
                bins.append(current)
                current = []
                running = 0
        if current:
            if bins:
                bins[-1].extend(current)
            else:
                bins.append(current)

        motif_to_bin: dict[int, str] = {}
        bin_order: list[str] = []
        for bin_lengths in bins:
            start_len = min(bin_lengths)
            end_len = max(bin_lengths)
            if start_len == end_len:
                label = f"{start_len}bp"
            else:
                label = f"{start_len}-{end_len}bp"
            bin_order.append(label)
            for length in bin_lengths:
                motif_to_bin[length] = label
        return motif_to_bin, bin_order

    # Per-class mutation rates (STR) and binning backbone
    str_rates = per_type_all[per_type_all["str_class"] == "STR_motif"].copy()
    str_rates["motif_len"] = str_rates["mutation_type"].str.extract(r"motif_(\d+)bp")[0]
    str_rates = str_rates.dropna(subset=["motif_len"]).copy()
    str_rates["motif_len"] = str_rates["motif_len"].astype(int)
    str_rates["indel_type"] = str_rates["mutation_type"].str.extract(r"^(ins|del)")
    str_rates = with_sample_label(str_rates)

    motif_counts = (
        str_rates.groupby("motif_len", as_index=False)["count"].sum().set_index("motif_len")
    )
    motif_bin_map, motif_bin_order = make_motif_bins(
        motif_counts["count"], STR_MOTIF_MIN_COUNT
    )

    # Callable bases (STR), aggregated by adaptive bins
    str_callable = callable_all[callable_all["mutation_type"].str.contains("motif_")].copy()
    str_callable["motif_len"] = str_callable["mutation_type"].str.extract(r"motif_(\d+)bp")[0]
    str_callable = str_callable.dropna(subset=["motif_len"]).copy()
    str_callable["motif_len"] = str_callable["motif_len"].astype(int)
    str_callable = (
        str_callable.groupby(["sample", "mutation_type"], as_index=False)["callable_bases"]
        .sum()
        .rename(columns={"callable_bases": "callable_bases_total"})
    )
    str_callable["indel_type"] = str_callable["mutation_type"].str.extract(r"^(ins|del)")
    str_callable["motif_len"] = str_callable["mutation_type"].str.extract(r"motif_(\d+)bp")[0].astype(int)
    str_callable["motif_bin"] = str_callable["motif_len"].map(motif_bin_map)
    str_callable = str_callable.dropna(subset=["motif_bin"]).copy()
    str_callable = with_sample_label(str_callable)

    str_callable_binned = (
        str_callable.groupby(
            ["sample", "sample_label", "indel_type", "motif_bin"], as_index=False
        )["callable_bases_total"].sum()
    )
    str_callable_binned["class_pct"] = (
        str_callable_binned["callable_bases_total"] / INTERROGATED_BASES_PER_SAMPLE * 100
    )

    plot_stacked_heatmap(
        str_callable_binned,
        value_col="class_pct",
        title="STR motif callable bases (% interrogated, binned)",
        filename="aim2_str_callable_pct_heatmap_binned",
        cbar_label="% of interrogated bases",
        index_col="motif_bin",
        index_order=motif_bin_order,
        log_scale=False,
    )

    # Per-class mutation rates (STR), aggregated by bins
    str_rates["motif_bin"] = str_rates["motif_len"].map(motif_bin_map)
    str_rates = str_rates.dropna(subset=["motif_bin"]).copy()

    str_rates_binned = (
        str_rates.groupby(
            ["sample", "sample_label", "indel_type", "motif_bin"], as_index=False
        )[["count", "callable_bases"]].sum()
    )
    str_rates_binned["frequency"] = (
        str_rates_binned["count"] / str_rates_binned["callable_bases"]
    )

    # Correction: scale by STR coverage per sample
    str_rates_binned["str_coverage_pct"] = str_rates_binned["sample"].map(
        STR_COVERAGE_PERCENT_BY_SAMPLE
    )
    str_rates_binned["str_coverage_frac"] = (
        str_rates_binned["str_coverage_pct"] / 100
    )
    if str_rates_binned["str_coverage_frac"].isna().any():
        missing = sorted(
            str_rates_binned[str_rates_binned["str_coverage_frac"].isna()][
                "sample"
            ].unique()
        )
        print(f"Missing STR coverage for samples: {missing}")
    str_rates_binned["frequency_corrected"] = (
        str_rates_binned["frequency"] / str_rates_binned["str_coverage_frac"]
    )

    plot_stacked_heatmap(
        str_rates_binned,
        value_col="frequency",
        title="STR motif mutation rates (raw, binned)",
        filename="aim2_str_rate_heatmap_binned_raw",
        cbar_label="Mutation rate (per base, log scale)",
        index_col="motif_bin",
        index_order=motif_bin_order,
        log_scale=True,
    )
    plot_stacked_heatmap(
        str_rates_binned,
        value_col="frequency_corrected",
        title="STR motif mutation rates (STR-corrected, binned)",
        filename="aim2_str_rate_heatmap_binned_corrected",
        cbar_label="Mutation rate (per base, log scale)",
        index_col="motif_bin",
        index_order=motif_bin_order,
        log_scale=True,
    )

    save_table(
        str_rates_binned[[
            "sample",
            "sample_label",
            "indel_type",
            "motif_bin",
            "count",
            "callable_bases",
            "frequency",
            "frequency_corrected",
        ]],
        "aim2_str_rate_binned_raw_vs_corrected",
    )

    # STR motif-length distribution (fraction within type), binned
    if not passed_all.empty and "type" in passed_all.columns:
        str_motif_col = "str_motif_length" if "str_motif_length" in passed_all.columns else None
        if str_motif_col:
            passed_str = passed_all.copy()
            if {"in_STR_region", "in_STR"}.issubset(passed_str.columns):
                str_mask = passed_str["in_STR_region"] & passed_str["in_STR"]
            elif "in_STR" in passed_str.columns:
                str_mask = passed_str["in_STR"]
            else:
                str_mask = pd.Series([False] * len(passed_str), index=passed_str.index)

            passed_str = passed_str[str_mask].copy()
            passed_str[str_motif_col] = pd.to_numeric(
                passed_str[str_motif_col], errors="coerce"
            )
            passed_str = passed_str.dropna(subset=[str_motif_col]).copy()
            passed_str["motif_len"] = passed_str[str_motif_col].astype(int)
            passed_str["motif_bin"] = passed_str["motif_len"].map(motif_bin_map)
            passed_str = passed_str.dropna(subset=["motif_bin"]).copy()

            str_counts = (
                passed_str.groupby(["sample", "type", "motif_bin"], as_index=False)
                .size()
                .rename(columns={"size": "count"})
            )
            str_counts["fraction"] = str_counts["count"] / str_counts.groupby(
                ["sample", "type"]
            )["count"].transform("sum")
            if "indel_type" not in str_counts.columns and "type" in str_counts.columns:
                str_counts["indel_type"] = str_counts["type"].map({"ins": "ins", "del": "del"})
            str_counts = with_sample_label(str_counts)

            plot_stacked_heatmap(
                str_counts,
                value_col="fraction",
                title="STR motif distribution (fraction within type, binned)",
                filename="aim2_str_motif_distribution_binned",
                cbar_label="Fraction within type",
                index_col="motif_bin",
                index_order=motif_bin_order,
                log_scale=False,
            )

**STR comparability notes:**
- STR per‑class rates use `per_type_mutation_frequency.tsv` (raw), normalized by class‑specific callable bases, so they are comparable across samples within the same motif class.
- STR‑corrected rates scale raw rates by per‑sample STR coverage (`repeat_coverage_percent`) to approximate the effect of applying the callable fraction only to STR bases.
- Callable‑bases % is shown against the full interrogated bases; the corrected view is expressed via the STR‑corrected rate.
- Motif‑length distributions are fractions within each sample and type, so they compare shape rather than absolute rate.

In [ ]:
**Interpretation (Aim 2):**
Non‑STR rate contrasts, size distributions, and tandem‑duplication fractions directly test whether insertion and deletion processes are mechanistically distinct.

**Comparability notes:**
- The per‑class rate heatmap uses `per_type_mutation_frequency.tsv`, which is normalized by class‑specific callable bases, so rates are comparable across samples within the same class.
- Size‑distribution heatmaps use fractions within each sample and type, so they compare shape rather than absolute rates.
- Tandem‑duplication fractions are computed within short insertions per sample, making them comparable across samples as proportions.

**Figure caption (Aim 3, adapted from Fig. 2c):** Mutation rate versus paternal age across donors. Points show sample-level raw mutation rates and the line indicates a linear regression fit with 95% confidence interval.

**Comparability note:** Rates are computed from observed indel counts over callable bases within a matched analysis scope per sample. This panel tests trend direction with age and is not, by itself, a class-specific mechanistic comparison.

## Aim 3 — Paternal age effect

Goal: compare mutation rates and insertion:deletion ratios across donors and relate to paternal age.

In [ ]:
# Fig. 2c-adapted: mutation rate vs paternal age (paper-style)
if "aim3_table" not in globals() or not isinstance(aim3_table, pd.DataFrame) or aim3_table.empty:
    if per_type_all.empty:
        print("No per-type data available for Fig. 2c-adapted panel.")
        aim3_rate_df = pd.DataFrame()
    else:
        tmp = per_type_all.copy()
        tmp["indel_type"] = tmp["mutation_type"].str.extract(r"^(ins|del)")
        sample_counts = tmp.groupby(["sample", "indel_type"], as_index=False)["count"].sum()
        sample_callable = tmp.groupby("sample", as_index=False)["callable_bases"].sum()
        rate_df = sample_counts.pivot_table(
            index="sample", columns="indel_type", values="count", aggfunc="sum"
        ).reset_index()
        rate_df = rate_df.merge(sample_callable, on="sample", how="left")
        rate_df = rate_df.rename(columns={"callable_bases": "callable_bases_total"})
        rate_df["total_indels"] = rate_df[[c for c in ["ins", "del"] if c in rate_df.columns]].sum(axis=1)
        rate_df["mutation_rate_raw"] = rate_df["total_indels"] / rate_df["callable_bases_total"]
        aim3_rate_df = rate_df.merge(SAMPLE_METADATA, on="sample", how="left")
else:
    aim3_rate_df = aim3_table.copy()

if not aim3_rate_df.empty:
    aim3_rate_df = with_sample_label(aim3_rate_df)
    fig2c_df = aim3_rate_df.dropna(subset=["paternal_age", "mutation_rate_raw"]).copy()

    if len(fig2c_df) >= 2:
        save_table(
            fig2c_df[["sample", "sample_label", "paternal_age", "mutation_rate_raw"]],
            "aim3_fig2c_rate_vs_age_data",
            ext="tsv",
        )

        fig, ax = plt.subplots(figsize=(5.8, 3.5))
        sns.regplot(
            data=fig2c_df,
            x="paternal_age",
            y="mutation_rate_raw",
            ci=95,
            scatter_kws={"s": 55, "color": sns.color_palette("viridis", 5)[3]},
            line_kws={"color": "black", "linewidth": 1.2},
            ax=ax,
        )
        for _, row in fig2c_df.iterrows():
            ax.text(
                row["paternal_age"] + 0.1,
                row["mutation_rate_raw"],
                str(row["sample_label"]),
                fontsize=8,
                va="center",
            )
        ax.set_xlabel("Paternal age")
        ax.set_ylabel("Rate")
        fig.suptitle("Mutation rate vs paternal age (raw)")
        fig.tight_layout()
        save_figure(fig, "aim3_rate_vs_age_raw")
        save_figure(fig, "aim3_fig2c_rate_vs_age_raw")
        plt.close(fig)
    else:
        print("Need at least 2 samples with paternal age and mutation rate for Fig. 2c-adapted panel.")

In [ ]:
if per_type_all.empty:
    print("No per-type data available for Aim 3.")
else:
    per_type_all = per_type_all.copy()
    per_type_all["indel_type"] = per_type_all["mutation_type"].str.extract(r"^(ins|del)")

    # Raw per-sample rates (all classes)
    per_sample_counts = (
        per_type_all.groupby(["sample", "indel_type"], as_index=False)["count"].sum()
    )
    per_sample_callable = (
        per_type_all.groupby("sample", as_index=False)["callable_bases"].sum()
    )

    per_sample_rates = per_sample_counts.pivot_table(
        index="sample", columns="indel_type", values="count", aggfunc="sum"
    ).reset_index()
    per_sample_rates = per_sample_rates.merge(per_sample_callable, on="sample", how="left")
    per_sample_rates = per_sample_rates.rename(columns={"callable_bases": "callable_bases_total"})

    per_sample_rates["total_indels"] = per_sample_rates[["ins", "del"]].sum(axis=1)
    per_sample_rates["mutation_rate_raw"] = (
        per_sample_rates["total_indels"] / per_sample_rates["callable_bases_total"]
    )
    per_sample_rates["ins_del_ratio_raw"] = per_sample_rates["ins"] / per_sample_rates["del"]

    # STR-only raw vs corrected rates
    str_rates = per_type_all[per_type_all["str_class"] == "STR_motif"].copy()
    str_rates["str_coverage_pct"] = str_rates["sample"].map(STR_COVERAGE_PERCENT_BY_SAMPLE)
    str_rates["str_coverage_frac"] = str_rates["str_coverage_pct"] / 100
    str_rates["frequency_corrected"] = str_rates["frequency"] / str_rates["str_coverage_frac"]

    str_rate_summary = (
        str_rates.groupby(["sample", "indel_type"], as_index=False)
        .agg(
            rate_raw=("frequency", "mean"),
            rate_corrected=("frequency_corrected", "mean"),
        )
    )
    if str_rate_summary.empty:
        str_rate_wide = per_sample_rates[["sample"]].copy()
        str_rate_wide["ins_rate_raw"] = np.nan
        str_rate_wide["del_rate_raw"] = np.nan
        str_rate_wide["ins_rate_corrected"] = np.nan
        str_rate_wide["del_rate_corrected"] = np.nan
    else:
        str_rate_wide = str_rate_summary.pivot_table(
            index="sample", columns="indel_type", values=["rate_raw", "rate_corrected"]
        )
        str_rate_wide = str_rate_wide.reset_index()
        if "sample" not in str_rate_wide.columns:
            if "index" in str_rate_wide.columns:
                str_rate_wide = str_rate_wide.rename(columns={"index": "sample"})
            else:
                str_rate_wide = per_sample_rates[["sample"]].merge(
                    str_rate_wide, left_index=True, right_index=True, how="left"
                )
        renamed_cols = []
        for col in str_rate_wide.columns:
            if col == "sample" or (isinstance(col, tuple) and col[0] == "sample"):
                renamed_cols.append("sample")
            else:
                metric, indel = col
                renamed_cols.append(f"{indel}_{metric}")
        str_rate_wide.columns = renamed_cols

    aim3_table = per_sample_rates.merge(SAMPLE_METADATA, on="sample", how="left")
    aim3_table = aim3_table.merge(str_rate_wide, on="sample", how="left")

    aim3_table = aim3_table[
        [
            "sample",
            "donor_id",
            "paternal_age",
            "callable_bases_total",
            "mutation_rate_raw",
            "ins_del_ratio_raw",
            "ins_rate_raw",
            "del_rate_raw",
            "ins_rate_corrected",
            "del_rate_corrected",
        ]
    ]
    save_table(aim3_table, "aim3_paternal_age_summary")
    display(aim3_table)

    # Overall mutation rate vs age (raw)
    age_plot_df = aim3_table.dropna(subset=["paternal_age", "mutation_rate_raw"]).copy()
    if len(age_plot_df) >= 2:
        fig, ax = plt.subplots(figsize=(5.8, 3.4))
        sns.regplot(
            data=age_plot_df,
            x="paternal_age",
            y="mutation_rate_raw",
            ax=ax,
            scatter_kws={"s": 50},
            line_kws={"color": "black"},
        )
        ax.set_xlabel("Paternal age")
        ax.set_ylabel("Mutation rate")
        fig.suptitle("Mutation rate vs paternal age (raw)")
        fig.tight_layout()
        save_figure(fig, "aim3_rate_vs_age_raw")
        plt.close(fig)
    else:
        print("Add paternal ages in SAMPLE_METADATA to enable the Aim 3 plot.")

    # STR-corrected rate vs age (use mean of ins/del corrected)
    age_plot_df = aim3_table.dropna(subset=["paternal_age", "ins_rate_corrected", "del_rate_corrected"]).copy()
    if len(age_plot_df) >= 2:
        age_plot_df["str_rate_corrected_mean"] = (
            age_plot_df["ins_rate_corrected"] + age_plot_df["del_rate_corrected"]
        ) / 2
        fig, ax = plt.subplots(figsize=(5.8, 3.4))
        sns.regplot(
            data=age_plot_df,
            x="paternal_age",
            y="str_rate_corrected_mean",
            ax=ax,
            scatter_kws={"s": 50},
            line_kws={"color": "black"},
        )
        ax.set_xlabel("Paternal age")
        ax.set_ylabel("STR-corrected rate")
        fig.suptitle("STR-corrected rate vs paternal age")
        fig.tight_layout()
        save_figure(fig, "aim3_str_corrected_rate_vs_age")
        plt.close(fig)

    # Insertion:deletion ratio vs age (raw)
    age_plot_df = aim3_table.dropna(subset=["paternal_age", "ins_del_ratio_raw"]).copy()
    if len(age_plot_df) >= 2:
        fig, ax = plt.subplots(figsize=(5.8, 3.4))
        sns.regplot(
            data=age_plot_df,
            x="paternal_age",
            y="ins_del_ratio_raw",
            ax=ax,
            scatter_kws={"s": 50},
            line_kws={"color": "black"},
        )
        ax.set_xlabel("Paternal age")
        ax.set_ylabel("Insertion:deletion ratio")
        fig.suptitle("Insertion:deletion ratio vs paternal age (raw)")
        fig.tight_layout()
        save_figure(fig, "aim3_ins_del_ratio_vs_age_raw")
        plt.close(fig)

    aim3_table

NameError: name 'per_type_all' is not defined

In [ ]:
# Non-STR 1–10bp mutation rate vs age
if per_type_all.empty:
    print("No per-type data available for non-STR 1–10bp rates.")
else:
    non_str = per_type_all.copy()
    non_str["len_bp"] = pd.to_numeric(non_str["len_or_motif"], errors="coerce")
    non_str = non_str[
        (non_str["str_class"] == "non_STR") & non_str["len_bp"].between(1, 10)
    ].copy()

    if non_str.empty:
        print("No non-STR 1–10bp rows found.")
    else:
        non_str_rates = (
            non_str.groupby("sample", as_index=False)[["count", "callable_bases"]].sum()
        )
        non_str_rates["non_str_1_10_rate"] = (
            non_str_rates["count"] / non_str_rates["callable_bases"]
        )
        non_str_rates = non_str_rates.merge(SAMPLE_METADATA, on="sample", how="left")
        non_str_rates = with_sample_label(non_str_rates)

        save_table(non_str_rates, "aim3_non_str_1_10_rate")
        display(
            non_str_rates[
                ["sample", "sample_label", "donor_id", "paternal_age", "non_str_1_10_rate"]
            ]
        )

        age_plot_df = non_str_rates.dropna(
            subset=["paternal_age", "non_str_1_10_rate"]
        ).copy()
        if len(age_plot_df) >= 2:
            fig, ax = plt.subplots(figsize=(5.8, 3.4))
            sns.regplot(
                data=age_plot_df,
                x="paternal_age",
                y="non_str_1_10_rate",
                ax=ax,
                scatter_kws={"s": 50},
                line_kws={"color": "black"},
            )
            ax.set_xlabel("Paternal age")
            ax.set_ylabel("Non-STR 1–10bp mutation rate")
            fig.suptitle("Non-STR 1–10bp mutation rate vs paternal age")
            fig.tight_layout()
            save_figure(fig, "aim3_non_str_1_10_rate_vs_age")
            plt.close(fig)
        else:
            print("Need at least 2 samples with ages to plot non-STR 1–10bp")

NameError: name 'per_type_all' is not defined

In [ ]:
# STR vs non-STR rates vs age (raw vs corrected STR)
if per_type_all.empty:
    print("No per-type data available for Aim 3 STR vs non-STR.")
else:
    rate_by_class = per_type_all.copy()
    rate_by_class["region_group"] = np.where(
        rate_by_class["str_class"] == "STR_motif", "STR", "non_STR"
    )

    rate_summary = (
        rate_by_class.groupby(["sample", "region_group"], as_index=False)
        .agg(rate_raw=("frequency", "mean"))
    )

    str_only = rate_by_class[rate_by_class["str_class"] == "STR_motif"].copy()
    str_only["str_coverage_pct"] = str_only["sample"].map(STR_COVERAGE_PERCENT_BY_SAMPLE)
    str_only["str_coverage_frac"] = str_only["str_coverage_pct"] / 100
    str_only["rate_corrected"] = str_only["frequency"] / str_only["str_coverage_frac"]

    str_corrected = (
        str_only.groupby("sample", as_index=False)
        .agg(rate_corrected=("rate_corrected", "mean"))
        .assign(region_group="STR_corrected")
    )

    rate_summary = rate_summary.merge(SAMPLE_METADATA, on="sample", how="left")
    rate_summary = with_sample_label(rate_summary)

    rate_summary = pd.concat([rate_summary, str_corrected.merge(SAMPLE_METADATA, on="sample", how="left")])
    rate_summary = with_sample_label(rate_summary)

    fig, ax = plt.subplots(figsize=(6.2, 3.8))
    sns.barplot(
        data=rate_summary,
        x="sample_label",
        y="rate_raw",
        hue="region_group",
        ax=ax,
    )
    ax.set_xlabel("Sample")
    ax.set_ylabel("Mutation rate")
    fig.suptitle("STR vs non-STR rates (raw + STR-corrected)")
    fig.tight_layout()
    save_figure(fig, "aim3_str_vs_non_str_rate")
    plt.close(fig)


In [ ]:
# Non-STR 1–10bp size profile vs age (heatmap)
if passed_all.empty:
    print("No passed indel data available for size profiles.")
else:
    passed_all = passed_all.copy()
    if "length" in passed_all.columns and "type" in passed_all.columns:
        passed_all["length"] = pd.to_numeric(passed_all["length"], errors="coerce")
        passed_all = passed_all.dropna(subset=["length"]).copy()
        passed_all["length"] = passed_all["length"].astype(int)
        passed_all["size_bin"] = passed_all["length"].apply(assign_size_bin)

        if {"in_STR_region", "in_STR"}.issubset(passed_all.columns):
            non_str_mask = ~(passed_all["in_STR_region"] & passed_all["in_STR"])
        elif "in_STR" in passed_all.columns:
            non_str_mask = ~passed_all["in_STR"]
        else:
            non_str_mask = pd.Series([True] * len(passed_all), index=passed_all.index)

        passed_non_str = passed_all[non_str_mask].copy()
        size_counts = (
            passed_non_str.groupby(["sample", "type", "size_bin"], as_index=False)
            .size()
            .rename(columns={"size": "count"})
        )
        size_counts["fraction"] = size_counts["count"] / size_counts.groupby(
            ["sample", "type"]
        )["count"].transform("sum")
        size_counts = with_sample_label(size_counts)

        size_counts = size_counts.merge(SAMPLE_METADATA, on="sample", how="left")
        size_counts = size_counts.sort_values(["paternal_age", "sample_label"])

        ins_size = size_counts[size_counts["type"] == "ins"].copy()
        del_size = size_counts[size_counts["type"] == "del"].copy()

        heatmap_ins = ins_size.pivot(
            index="size_bin",
            columns="sample_label",
            values="fraction",
        )
        heatmap_del = del_size.pivot(
            index="size_bin",
            columns="sample_label",
            values="fraction",
        )

        combined_min = np.nanmin(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )
        combined_max = np.nanmax(
            pd.concat([heatmap_ins.stack(), heatmap_del.stack()], axis=0)
        )

        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(6.8, 6.6),
            sharex=True,
            gridspec_kw={"hspace": 0.06},
        )
        fig.subplots_adjust(right=0.86)
        cbar_ax = fig.add_axes([0.88, 0.16, 0.025, 0.68])

        sns.heatmap(
            heatmap_del,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=False,
            ax=axes[0],
        )
        axes[0].set_title("")
        axes[0].set_ylabel("Deletions")
        axes[0].set_xlabel("")

        sns.heatmap(
            heatmap_ins,
            cmap="viridis",
            linewidths=0.2,
            linecolor="white",
            vmin=combined_min,
            vmax=combined_max,
            cbar=True,
            cbar_ax=cbar_ax,
            cbar_kws={"label": "Fraction within type"},
            ax=axes[1],
        )
        axes[1].set_title("")
        axes[1].set_ylabel("Insertions")
        axes[1].set_xlabel("Sample")

        fig.suptitle("Non-STR 1–10bp size profile vs age")
        fig.tight_layout(rect=[0, 0, 0.86, 0.96])
        save_figure(fig, "aim3_non_str_size_profile_vs_age")
        plt.close(fig)


**Aim 3 comparability notes:**
- Raw rates are normalized by class-specific callable bases, so comparisons are valid across samples within the same class.
- STR-corrected rates scale raw STR rates by per-sample STR coverage (`repeat_coverage_percent`), approximating STR-only denominators.
- Size-profile plots compare proportions (shape) rather than absolute rates.

**Interpretation (Aim 3):**
The rate-versus-age comparison and insertion:deletion ratios test whether the donors show the expected paternal age effect trend.